# Notebook 02 — Data cleaning 

1. Objectives

This notebook applies the cleaning decisions established during profiling (Notebook 01) to the four ENERGICAL datasets — transactions, orders, customers, and catalogue — preparing them for PostgreSQL integration.

Operations performed: data type standardization, duplicate removal, handling of critical missing values, text normalization, and enrichment of transaction records via the product catalogue.

2. Load Raw Data

In [6]:
import pandas as pd

In [7]:
transactions=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\transactions_stage V3 (1).csv")
orders=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\orders_stage_V2 (1).csv")
customers=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\customers_stage-V2.csv")
catalogue=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\catalogue_stage_V3.csv")



In [8]:
datasets = {
    "Transactions": transactions,
    "Orders": orders,
    "Customers": customers,
    "Catalogue": catalogue,
}

3. Cleaning functions

In [9]:
#cleaning data types
def clean_dtypes(df):
    for col in df.columns:
        if "date" in col:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    
    
    numeric_cols = ["quantity", "unit_price", "line_total",
                    "order_total_amount", "total_quantity", "n_lines","shipping_cost","Prix unitaire"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("string")

    return df


In [10]:
#cleaning duplicates
def clean_duplicates(df):
    before=len(df)
    df = df.drop_duplicates()
    after=len(df)
    print(f"removed {before-after} duplicates")
    return df

In [11]:
#cleaning missing values
def clean_missing_values(df, df_name):
    before=len(df)
    if df_name == "transactions":
        df = df.dropna(subset=["quantity", "unit_price", "line_total"])  
    after=len(df) 
    print(f"removed {before-after} incomplete rows")
    return df
        

In [12]:
#Business rules
def validate_business_rules(df, df_name):

    print(f"\nBusiness Rule Validation - {df_name}")

    if "quantity" in df.columns:
        print(f"Negative quantities: {(df['quantity'] < 0).sum()}")

    if "unit_price" in df.columns:
        print(f"Negative prices: {(df['unit_price'] < 0).sum()}")
    if "line_total" in df.columns:
        print(f"Negative line totals: {(df['line_total'] < 0).sum()}")

    if "order_date" in df.columns:
        print(f"Future dates: {(df['order_date'] > pd.Timestamp.today()).sum()}")
    
    return df
        

In [13]:
#Clean product name
def clean_product_names(df):

    if "product_name" in df.columns:

        df["product_name"] = (
            df["product_name"]
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    return df

4. Transactions Cleaning

In [14]:
clean_transactions=transactions.copy()

In [15]:
clean_transactions.drop(columns=["payment_method_group"], inplace=True)
clean_transactions.rename(
    columns={
        "Code Client": "customer_id_stage",
        "Titre moyen du paiement": "payment_method",
        "Titre de la méthode d’expédition": "shipping_method",
        "Montant total de la commande": "order_total_amount",
        "Montant de l’expédition commande": "shipping_cost",
        "Poids total": "total_weight"
    },
    inplace=True
)
clean_transactions["shipping_cost"] = (
    clean_transactions["shipping_cost"]
    .str.replace(" DA", "", regex=False)
    .str.replace(",", "", regex=False)
)
clean_transactions["order_total_amount"] = (
    clean_transactions["order_total_amount"]
    .str.replace(" DA", "", regex=False)
    .str.replace(",", "", regex=False)
)
clean_transactions=clean_dtypes(clean_transactions)


In [16]:
clean_transactions["payment_method"] = clean_transactions["payment_method"].replace({
    "Autre": "Other",
    "Paiement à la livraison": "Cash on Delivery",
    "Versement CCP": "CCP Transfer",
    "Virement / Versement bancaire": "Bank Transfer",
    "Paiements par chèque": "Cheque Payment",
    "Paiement par chèque": "Cheque Payment",
    "Carte CIB & EDAHABIA": "CIB & Edahabia Card",
    "Carte CIB &amp; EDAHABIA": "CIB & Edahabia Card",
    "CIB / EDAHABIA": "CIB & Edahabia Card",
    "Espèces - Siège Social": "Cash - Siège Social",
    "Espèces - Bureau E-com": "Cash - E-commerce Office",
    "Espèces - Bureau Ecom": "Cash - E-commerce Office",
    "Espèces - Point d'enlèvement": "Cash - Pickup Point",
    "Other": "Other"
})

In [17]:
catalogue = catalogue.rename(columns={
    "SkU": "sku",
    "Nom": "product_name",
    "categorie": "category",
    "Sous-catégorie": "subcategory",
    "Prix unitaire": "unit_price"
})

In [18]:
clean_transactions=clean_duplicates(clean_transactions)
clean_transactions=clean_product_names(clean_transactions)
clean_transactions=clean_missing_values(clean_transactions,"transactions")
#Enrichment & merging 
clean_transactions = clean_transactions.merge(
    catalogue[["sku", "product_name", "subcategory"]],
    on="sku", how="left", suffixes=("", "_catalogue")
)
clean_transactions["product_name"] = clean_transactions["product_name"].fillna(clean_transactions["product_name_catalogue"])
clean_transactions["subcategory"] = clean_transactions["subcategory"].fillna(clean_transactions["subcategory_catalogue"])
clean_transactions["has_negative_price"] = clean_transactions["unit_price"] < 0
clean_transactions = clean_transactions.drop(columns=["product_name_catalogue", "subcategory_catalogue"])
print(f"Flagged as has_negative_price: {clean_transactions['has_negative_price'].sum()}")

clean_transactions["subcategory"] = clean_transactions["subcategory"].fillna("Unknown")
clean_transactions["product_name"] = clean_transactions["product_name"].fillna("Unknown")

removed 12 duplicates
removed 125 incomplete rows
Flagged as has_negative_price: 65


In [19]:
clean_transactions["subcategory"].eq("Unknown").sum()

np.int64(4256)

In [20]:
all_shipping_methods=clean_transactions["shipping_method"].drop_duplicates()
all_shipping_methods.to_csv("../../data/CleanData/all_shipping_methods.csv", index=False)



In [21]:

clean_transactions["free_shipping"] = clean_transactions["shipping_method"].str.contains(
    "gratuit|gratuite|free|مجانا",
    case=False,
    na=False
)

In [22]:
# Home Delivery
clean_transactions.loc[
    clean_transactions["shipping_method"].str.contains(
        "domicile|home|المنزل",
        case=False,
        na=False
    ),
    "shipping_method"
] = "Home Delivery"

In [23]:
# Pickup Point
clean_transactions.loc[
    clean_transactions["shipping_method"].str.contains(
        "point de retrait|pick|نقطة",
        case=False,
        na=False
    ),
    "shipping_method"
] = "Pickup Point"

In [24]:
# E-commerce Office
clean_transactions.loc[
    clean_transactions["shipping_method"].str.contains(
        "Bureau|Expédition",
        case=False,
        na=False
    ),
    "shipping_method"
] = "E-commerce Office"

In [25]:

# Customer Pickup
clean_transactions.loc[
    clean_transactions["shipping_method"].str.contains(
        "point d'enlèvement|point d'enlévement|ADRAR|ALGER|BISKRA|BLIDA|BECHAR|CONSTANTINE|CONSTANTIN|DAR EL BEIDA|EL EULMA|BARIKA|GHARDAIA|ORAN|OUARGLA",
        case=False,
        na=False
    ),
    "shipping_method"
] = "Collection Point" 

# International
clean_transactions.loc[
    clean_transactions["shipping_method"].str.contains(
        "International",
        case=False,
        na=False
    ),
    "shipping_method"
] = "EMS International" 

In [26]:
valid_methods = [
    "Home Delivery",
    "Pickup Point",
    "Collection Point",
    "E-commerce Office",
    "EMS International"
]

clean_transactions.loc[
    ~clean_transactions["shipping_method"].isin(valid_methods),
    "shipping_method"
] = "Unknown"

In [27]:
clean_transactions["shipping_method"].drop_duplicates()

0                 Unknown
3           Home Delivery
105          Pickup Point
827      Collection Point
995     E-commerce Office
4314    EMS International
Name: shipping_method, dtype: string

In [28]:
clean_transactions["payment_method"] = clean_transactions["payment_method"].fillna("Unknown")


5. Catalogue Cleaning

In [29]:
clean_catalogue=catalogue.copy()
clean_catalogue=clean_dtypes(clean_catalogue)
clean_catalogue=clean_duplicates(clean_catalogue)
clean_catalogue=clean_product_names(clean_catalogue)

# Flag which catalogue products were actually sold
clean_catalogue["ever_sold"] = clean_catalogue["sku"].isin(clean_transactions["sku"].unique())

# Recover price for SKUs with transaction history
missing_price_skus = clean_catalogue[clean_catalogue["unit_price"].isna()]["sku"]

recovered_prices = (
    clean_transactions[clean_transactions["sku"].isin(missing_price_skus)]
    .groupby("sku")["unit_price"]
    .median()
)

clean_catalogue["unit_price"] = clean_catalogue["unit_price"].fillna(
    clean_catalogue["sku"].map(recovered_prices)
)

print(f"Remaining missing unit_price: {clean_catalogue['unit_price'].isna().sum()}")

removed 93 duplicates
Remaining missing unit_price: 215


In [30]:
clean_catalogue["subcategory"]=clean_catalogue["subcategory"].fillna("unknown")

In [31]:
clean_catalogue["subcategory"] = (
    clean_catalogue["subcategory"]
    .str.split(">")
    .str[1]
    .str.strip()
)

6. Customers Cleaning

In [32]:
clean_customers=customers.copy()
clean_customers=clean_duplicates(clean_customers)
clean_customers=clean_dtypes(clean_customers)

removed 0 duplicates


7. Orders Cleaning

In [33]:
clean_orders=orders.copy()
clean_orders=clean_dtypes(clean_orders)
clean_orders=clean_duplicates(clean_orders)
clean_orders=clean_product_names(clean_orders)

removed 0 duplicates


8. Validation

In [34]:
cleaned = {
    "Transactions": clean_transactions,
    "Orders": clean_orders,
    "Customers": clean_customers,
    "Catalogue": clean_catalogue
}

In [35]:
for name, df in cleaned.items():
    print(name)
    print(df.info())
    print(df.isna().sum())
    print(df.duplicated().sum())
    validate_business_rules(df,name)

Transactions
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20274 entries, 0 to 20273
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id_stage                 20274 non-null  string        
 1   customer_id_stage              20274 non-null  string        
 2   order_date                     20274 non-null  datetime64[ns]
 3   wilaya_raw                     20274 non-null  string        
 4   wilaya_normalized              20274 non-null  string        
 5   geo_quality_flag               20274 non-null  string        
 6   customer_type_inferred         20274 non-null  string        
 7   sku                            20274 non-null  object        
 8   product_name                   20274 non-null  string        
 9   sku_quality                    20274 non-null  string        
 10  category                       20274 non-null  string        
 11  su

In [36]:
clean_transactions["shipping_method"].value_counts(dropna=False)

shipping_method
Home Delivery        9482
Pickup Point         7667
Unknown              1559
E-commerce Office    1378
Collection Point      185
EMS International       3
Name: count, dtype: Int64

In [37]:
clean_transactions["payment_method"].value_counts(dropna=False)

payment_method
Cash on Delivery                                         16917
CIB & Edahabia Card                                       1515
Cash - E-commerce Office                                  1321
Cheque Payment                                             181
Virement / Versement  bancaire                             109
Cash - Siège Social                                         64
CCP Transfer                                                63
Virement / Versement  bancaire / Paiements par chèque       47
Cash on delivery                                            27
Unknown                                                     10
Other                                                        8
Versement espèces                                            7
Cash - Pickup Point                                          5
Name: count, dtype: Int64

9. Export Clean Datasets

In [38]:
clean_transactions.to_csv("../../data/CleanData/clean_transactions.csv", index=False)
clean_orders.to_csv("../../data/CleanData/clean_orders.csv", index=False)
clean_customers.to_csv("../../data/CleanData/clean_customers.csv", index=False)
clean_catalogue.to_csv("../../data/CleanData/clean_catalogue.csv", index=False)


10. Cleaning Summary

In [39]:
raw_datasets = {
    "Transactions": transactions,
    "Orders": orders,
    "Customers": customers,
    "Catalogue": catalogue,
}

summary_rows = []
for name, clean_df in cleaned.items():
    raw_rows = len(raw_datasets[name])
    clean_rows = len(clean_df)
    removed = raw_rows - clean_rows
    pct_removed = round((removed / raw_rows) * 100, 2)
    summary_rows.append({
        "Dataset": name,
        "Rows (raw)": raw_rows,
        "Rows (clean)": clean_rows,
        "Rows removed": removed,
        "% removed": pct_removed
    })

cleaning_summary = pd.DataFrame(summary_rows)
cleaning_summary

,Dataset,Rows (raw),Rows (clean),Rows removed,% removed
0,Transactions,20405,20274,131,0.64
1,Orders,9245,9245,0,0.00
2,Customers,5776,5776,0,0.00
3,Catalogue,3850,3757,93,2.42



The data cleaning process focused on improving the consistency, completeness, and reliability of the ENERGICAL datasets before database integration. The following operations were performed:

- Standardized data types (dates, numeric values, and categorical variables).
- Removed duplicate records.
- Removed records with missing values in critical business fields where no reliable recovery was possible.
- Cleaned and normalized text fields (whitespace, formatting, and naming inconsistencies).
- Standardized product names where applicable.
- Enriched transaction data through SKU-based matching with the product catalogue to recover missing product names and subcategories whenever a reliable match existed.
- Recovered missing catalogue prices for products with transaction history using the median observed transaction price.
- Standardized payment methods by consolidating inconsistent labels into a common set of business categories.
- Standardized shipping methods by consolidating multilingual and inconsistently formatted values into standardized delivery categories while preserving free-shipping information in a separate boolean field.
- Created additional business features to support downstream analytics, including has_negative_price, free_shipping, ever_sold, and standardized payment and shipping method fields.
- Performed business rule validation to identify anomalies such as negative transaction values, missing catalogue prices, and other records requiring manual review.
- Validated the cleaned datasets by checking data types, duplicates, missing values, referential consistency, and overall data integrity.

The cleaned datasets are now ready for PostgreSQL integration and subsequent analytical tasks. Detailed statistics, cleaning decisions, and identified data quality issues are documented in the accompanying Data Cleaning Report.